## **Figure-2-Fetch-Chromatogram-Data**

In this notebook load the data into pandas dataframes so it can be processed by an enviroment without massdash. 

In [1]:
from massdash.loaders import OpenSwathXICParquetLoader
from massdash.loaders import SqMassLoader
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import sys
import massdash
%matplotlib inline

In [2]:
print("MassDash Version", massdash.__version__)
print("Python Version", sys.version)

MassDash Version 0.1.0
Python Version 3.11.5 (main, Sep 18 2023, 12:23:42) [GCC 12.3.1 20230526]


In [3]:
prec = 'ENSIEILSSTIK'
chg = 2

In [4]:
gpf_experiment = OpenSwathXICParquetLoader(rsltsFile="../../results/K562-Refine-PeptDeep-NoMods-With-GPF/osw/pyprophet_XGB/merged.oswpqd/RostDIA640_SP2um_90min_250ngK562_100nL_Slot1-5_1_1320_6-27-2021.oswpq/",
                                           dataFiles="../../results/K562-Refine-PeptDeep-NoMods-With-GPF/osw/RostDIA640_SP2um_90min_250ngK562_100nL_Slot1-5_1_1320_6-27-2021/oswOut/RostDIA640_SP2um_90min_250ngK562_100nL_Slot1-5_1_1320_6-27-2021.sqMass_xic.parquet")

in results loader
[2026-03-24 16:15:42,213] GenericResultsAccess - INFO - Initialized lazy datasets for OSWPQ data: ../../results/K562-Refine-PeptDeep-NoMods-With-GPF/osw/pyprophet_XGB/merged.oswpqd/RostDIA640_SP2um_90min_250ngK562_100nL_Slot1-5_1_1320_6-27-2021.oswpq/


In [5]:
full_run_initial = OpenSwathXICParquetLoader(rsltsFile="../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/",
                                           dataFiles="../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/oswOut/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.sqMass_xic.parquet")

in results loader
[2026-03-24 16:15:43,013] GenericResultsAccess - INFO - Initialized lazy datasets for OSWPQ data: ../../results/K562-PeptDeep-NoMods-Analysis/osw/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021/pyprophet_XGB/Rost_DIApy3_SP2um_90min_250ngK562_100nL_1_Slot1-5_1_1330_6-28-2021.oswpq/


##### **Load GPF Chromatogram**

In [6]:
gpf_chromatograms = gpf_experiment.loadTransitionGroupsDf(prec, 2, )

gpf_chromatograms["intensity"] = (
    gpf_chromatograms
    .groupby("annotation")["intensity"]
    .transform(lambda x: savgol_filter(x, window_length=11, polyorder=3))
)


gpf_chromatograms_only_ms2 = gpf_chromatograms[
    (~gpf_chromatograms['annotation'].str.contains('Precursor'))]

In [7]:
gpf_chromatograms_only_ms2.to_pickle("gpf_chromatogram.pkl")

##### **Load Reg Chromatogram**

In [8]:
reg_chromatograms = full_run_initial.loadTransitionGroupsDf(prec, 2)

reg_chromatograms["intensity"] = (
    reg_chromatograms
    .groupby("annotation")["intensity"]
    .transform(lambda x: savgol_filter(x, window_length=11, polyorder=3))
)

reg_chromatograms_only_ms2 = reg_chromatograms[
    (~reg_chromatograms['annotation'].str.contains('Precursor'))]


In [9]:
reg_chromatograms_only_ms2.to_pickle("reg_chromatogram.pkl")